In [1]:
!pip install cupy-cuda12x # или cupy-cuda11x в зависимости от версии драйвера

In [21]:
import numpy as np
import re
from math import tan, radians, sqrt
import numba
from numba import jit, prange
import matplotlib.pyplot as plt
from PIL import Image
import cupy as cp # Импортируем CuPy вместо Numba

In [26]:
# --- Парсеры остаются без изменений (они работают на CPU и быстрые) ---
def parse_angle_input(angle_str: str) -> float:
    angle_str = angle_str.strip().lower().replace('π', 'pi')
    if 'pi' in angle_str:
        angle_str = angle_str.replace(' ', '')
        if angle_str == 'pi': return np.pi
        elif angle_str == '-pi': return -np.pi
        if '/' in angle_str:
            if ' ' in angle_str.split('/')[0]:
                num_part, denom = angle_str.split('/')
                a = num_part.replace(' pi', '').replace('pi ', '')
                if a == '': a = '1'
                result = float(a) * np.pi / float(denom)
            else:
                _, denom = angle_str.split('/')
                result = np.pi / float(denom)
            return result
        if ' ' in angle_str:
            parts = [p for p in angle_str.split(' ') if p]
            if 'pi' in parts:
                for p in parts:
                    if p != 'pi':
                        try: return float(p) * np.pi
                        except ValueError: pass
            return float(parts[0]) * np.pi
    try:
        return float(angle_str) * np.pi
    except ValueError:
        raise ValueError(f"Невозможно распознать угол: {angle_str}")

def parse_H_input(H_str: str) -> float:
    H_str = H_str.strip().lower()
    if H_str in ['inf', '+inf']: return np.inf
    elif H_str == '-inf': return -np.inf
    if H_str.startswith('tan('):
        match = re.match(r'tan\((.*)\)', H_str)
        if match:
            arg_str = match.group(1)
            if 'deg' in arg_str:
                return tan(radians(float(arg_str.replace('deg', ''))))
            else:
                arg = parse_angle_input(arg_str) if 'pi' in arg_str else float(arg_str)
                return tan(arg)
    return float(H_str)

class Gyrator:
    """Класс для гираторного преобразования на GPU с использованием CuPy."""

    def __init__(self, size: int = 512, scale: float = 2.0):
        self.size = size
        self.scale = scale

        # Создаем координаты сразу на GPU
        # Используем linspace от -scale до scale
        self.x_gpu = cp.linspace(-scale, scale, size, endpoint=False, dtype=cp.float64)
        self.y_gpu = cp.linspace(-scale, scale, size, endpoint=False, dtype=cp.float64)

        # Шаг дискретизации (может понадобиться для нормировки, если требуется строгая физическая модель)
        self.dx = 2 * scale / size
        self.dy = 2 * scale / size

    def transform(self, field: np.ndarray, alpha: float) -> np.ndarray:
        """Гираторное преобразование на GPU."""

        # 1. Перенос данных на GPU
        field_gpu = cp.asarray(field, dtype=cp.complex128)

        sin_alpha = cp.sin(alpha)

        # Обработка особых случаев (alpha ≈ 0 или pi)
        if cp.abs(sin_alpha) < 1e-10:
            cos_alpha = cp.cos(alpha)
            if cp.abs(cos_alpha - 1) < 1e-10:
                return field.copy()  # Тождественное преобразование
            else:
                # Поворот на 180 градусов (инверсия)
                return cp.asnumpy(cp.flipud(cp.fliplr(field_gpu)))

        cos_alpha = cp.cos(alpha)
        cot_alpha = cos_alpha / sin_alpha
        csc_alpha = 1.0 / sin_alpha

        size = self.size
        const_2pi = 2.0 * cp.pi

        # Масштабный коэффициент (важно!)
        scale_factor = 1.0 / cp.abs(sin_alpha) * self.dx * self.dy

        # Координаты
        x_src = self.x_gpu  # (N,) - xm
        y_src = self.y_gpu  # (N,) - yn

        result_gpu = cp.zeros((size, size), dtype=cp.complex128)

        # Предвычисляем матрицу фаз exp(1j*2π*cot*xm*yn) для всех m,n
        phase_cot_mn = cp.exp(1j * const_2pi * cot_alpha * cp.outer(x_src, y_src))  # (N, N)

        # Основной цикл по строкам результата (по координате u)
        for i in range(size):
            u = x_src[i]

            # Фаза, зависящая от yn и u: exp(-1j*2π*csc*yn*u)
            phase_csc_yn_u = cp.exp(-1j * const_2pi * csc_alpha * y_src * u)  # (N,)

            # Inner[m] = Sum_n [field[m,n] * phase_cot_mn[m,n] * phase_csc_yn_u[n]]
            A = field_gpu * phase_cot_mn  # (N, N)
            Inner = cp.dot(A, phase_csc_yn_u)  # (N,) - сумма по n для каждого m

            # Фаза exp(-1j*2π*csc*xm*v) для всех m,v
            phase_csc_xm_v = cp.exp(-1j * const_2pi * csc_alpha * cp.outer(x_src, y_src))  # (N, N)

            # Sum_m [Inner[m] * phase_csc_xm_v[m,v]] для всех v
            sum_over_m = cp.dot(Inner, phase_csc_xm_v)  # (N,)

            # Финальная фаза: exp(1j*2π*cot*u*v)
            phase_cot_uv = cp.exp(1j * const_2pi * cot_alpha * u * y_src)  # (N,)

            # Итоговый результат для строки i
            result_row = scale_factor * sum_over_m * phase_cot_uv

            result_gpu[i, :] = result_row

        # Перенос обратно на CPU
        return cp.asnumpy(result_gpu)

In [30]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from math import factorial
# Добавлен импорт j0 для пучка Бесселя
from scipy.special import hermite, j0

# ============================================================================
# ФУНКЦИИ ДЛЯ ГЕНЕРАЦИИ РАЗЛИЧНЫХ ПУЧКОВ
# ============================================================================
def gaussian_beam_with_cubic_phase(size, w0, a, b, center=None):
    if center is None: center = (size // 2, size // 2)
    x = np.arange(size) - center[0]
    y = np.arange(size) - center[1]
    x, y = np.meshgrid(x, y)
    r_sq = x**2 + y**2
    gaussian = np.exp(-r_sq / (w0**2))
    cubic_phase = np.exp(1j * (a * x**3 + b * y**3))
    return gaussian * cubic_phase

def gaussian_beam_with_quadratic_phase(size, w0, c, d, center=None):
    if center is None: center = (size // 2, size // 2)
    x = np.arange(size) - center[0]
    y = np.arange(size) - center[1]
    x, y = np.meshgrid(x, y)
    r_sq = x**2 + y**2
    gaussian = np.exp(-r_sq / (w0**2))
    quadratic_phase = np.exp(1j * (c * x**2 + d * y**2))
    return gaussian * quadratic_phase

def simple_gaussian_beam(size, w0, center=None):
    if center is None: center = (size // 2, size // 2)
    x = np.arange(size) - center[0]
    y = np.arange(size) - center[1]
    x, y = np.meshgrid(x, y)
    r_sq = x**2 + y**2
    return np.exp(-r_sq / (w0**2))

def plane_wave_with_linear_phase(size, kx, ky, center=None):
    if center is None: center = (size // 2, size // 2)
    x = np.arange(size) - center[0]
    y = np.arange(size) - center[1]
    x, y = np.meshgrid(x, y)
    return np.exp(1j * (kx * x + ky * y))

def hyperbolic_phase_wave(size, c, center=None):
    if center is None: center = (size // 2, size // 2)
    x = np.arange(size) - center[0]
    y = np.arange(size) - center[1]
    x, y = np.meshgrid(x, y)
    return np.exp(1j * 2 * np.pi * c * x * y)

def spherical_wave(size, b, center=None):
    if center is None: center = (size // 2, size // 2)
    x = np.arange(size) - center[0]
    y = np.arange(size) - center[1]
    x, y = np.meshgrid(x, y)
    r_sq = x**2 + y**2
    return np.exp(-1j * np.pi * b * r_sq)

def hermit_gaussian_mode(size, m, n, w0=30, center=None):
    if center is None: center = (size // 2, size // 2)
    x = np.arange(size) - center[0]
    y = np.arange(size) - center[1]
    x, y = np.meshgrid(x, y)
    r_sq = x**2 + y**2
    gaussian = np.exp(-r_sq / (w0**2))
    Hm = hermite(m)
    Hn = hermite(n)
    norm = 1.0 / np.sqrt(2**m * factorial(m) * 2**n * factorial(n) * np.pi)
    hermite_part = Hm(np.sqrt(2) * x / w0) * Hn(np.sqrt(2) * y / w0)
    return norm * hermite_part * gaussian

def circle_function(size, radius, center=None):
    if center is None: center = (size // 2, size // 2)
    x = np.arange(size) - center[0]
    y = np.arange(size) - center[1]
    x, y = np.meshgrid(x, y)
    r_sq = x**2 + y**2
    circle = np.zeros_like(r_sq, dtype=complex)
    circle[r_sq <= radius**2] = 1.0 + 0j
    return circle

def laguerre_gaussian_mode(size, p, l, w0=30, center=None):
    if center is None: center = (size // 2, size // 2)
    x = np.arange(size) - center[0]
    y = np.arange(size) - center[1]
    X, Y = np.meshgrid(x, y)
    r = np.sqrt(X**2 + Y**2)
    phi = np.arctan2(Y, X)
    rho = np.sqrt(2) * r / w0
    rho2 = 2 * r**2 / w0**2
    k = abs(l)
    fact_pk = factorial(p + k)
    L_val = np.zeros_like(rho2)
    for m in range(p + 1):
        coef = ((-1)**m * fact_pk) / (factorial(p - m) * factorial(k + m) * factorial(m))
        L_val += coef * (rho2 ** m)
    amplitude = (rho**k) * L_val * np.exp(-r**2 / w0**2)
    phase = np.exp(-1j * l * phi)
    return amplitude * phase

def bessel_beam(size, kr, center=None):
    """
    Генерирует пучок Бесселя (нулевого порядка).
    Это недифрагирующий пучок, интенсивность которого описывается функцией J0.
    """
    if center is None: center = (size // 2, size // 2)
    x = np.arange(size) - center[0]
    y = np.arange(size) - center[1]
    x, y = np.meshgrid(x, y)
    r = np.sqrt(x**2 + y**2)
    # j0 - функция Бесселя первого рода нулевого порядка
    return j0(kr * r)

def superpose_beams(beams, weights):
    """
    Выполняет суперпозицию нескольких комплексных полей.
    beams: список 2D массивов (полей)
    weights: список весов (комплексных чисел)
    """
    if len(beams) == 0:
        return np.zeros((64, 64), dtype=complex) # Fallback

    # Инициализируем результат нулями той же формы
    result = np.zeros_like(beams[0], dtype=complex)

    for beam, w in zip(beams, weights):
        result += beam * w

    return result

# ============================================================================
# ФУНКЦИИ ВИЗУАЛИЗАЦИИ
# ============================================================================
def visualize_beam(beam, title, is_complex=True):
    plt.figure(figsize=(10, 8))
    if is_complex:
        plt.subplot(1, 2, 1)
        plt.imshow(np.abs(beam), cmap='gray')
        plt.title(f"{title} (Амплитуда)")
        plt.colorbar()
        plt.subplot(1, 2, 2)
        plt.imshow(np.angle(beam), cmap='hsv', vmin=-np.pi, vmax=np.pi)
        plt.title(f"{title} (Фаза)")
        plt.colorbar()
    else:
        plt.imshow(beam, cmap='gray')
        plt.title(title)
        plt.colorbar()
    plt.tight_layout()
    plt.show()

def visualize_comparison(original, transformed, alpha, params_str=""):
    ax3 = plt.subplot(1, 2, 1)
    im3 = ax3.imshow(np.abs(transformed), cmap='gray')
    ax3.set_title(f"Гираторное преобразование (Амплитуда)\nα={alpha/np.pi:.3f}π")
    plt.colorbar(im3, ax=ax3)
    ax4 = plt.subplot(1, 2, 2)
    im4 = ax4.imshow(np.angle(transformed), cmap='hsv', vmin=-np.pi, vmax=np.pi)
    ax4.set_title(f"Гираторное преобразование (Фаза)\nα={alpha/np.pi:.3f}π")
    plt.colorbar(im4, ax=ax4)
    plt.tight_layout()
    plt.show()

def visualize_12_angles(beam, gyrator, params_str=""):
    angles = np.linspace(0, np.pi/2, 12)
    fig_amp, axes_amp = plt.subplots(3, 4, figsize=(16, 12))
    fig_phase, axes_phase = plt.subplots(3, 4, figsize=(16, 12))
    fig_amp.suptitle(f"Амплитуды гираторного преобразования (12 углов от 0 до π/2)\n{params_str}", fontsize=14, fontweight='bold')
    fig_phase.suptitle(f"Фазы гираторного преобразования (12 углов от 0 до π/2)\n{params_str}", fontsize=14, fontweight='bold')
    axes_amp_flat = axes_amp.flatten()
    axes_phase_flat = axes_phase.flatten()
    for idx, alpha in enumerate(angles):
        transformed = gyrator.transform(beam, alpha)
        im_amp = axes_amp_flat[idx].imshow(np.abs(transformed), cmap='gray')
        axes_amp_flat[idx].set_title(f"α={alpha/np.pi:.3f}π\nH={np.tan(alpha):.3f}")
        axes_amp_flat[idx].axis('off')
        im_phase = axes_phase_flat[idx].imshow(np.angle(transformed), cmap='hsv', vmin=-np.pi, vmax=np.pi)
        axes_phase_flat[idx].set_title(f"α={alpha/np.pi:.3f}π\nH={np.tan(alpha):.3f}")
        axes_phase_flat[idx].axis('off')
    fig_amp.colorbar(im_amp, ax=axes_amp, orientation='vertical', fraction=0.02, pad=0.04)
    fig_phase.colorbar(im_phase, ax=axes_phase, orientation='vertical', fraction=0.02, pad=0.04)
    plt.tight_layout()
    plt.show()
    return angles

def save_beam_data_as_png(original, transformed, prefix, output_dir="results"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    print(f"  Создана папка для результатов: {output_dir}")

    # Проверка на корректность данных
    if original is None or transformed is None:
        print("Ошибка: Данные для сохранения отсутствуют (None).")
        return

    # Принудительная конвертация в numpy, если это cupy array
    if hasattr(original, 'get'):
        original = original.get()
    if hasattr(transformed, 'get'):
        transformed = transformed.get()

    # Вычисление амплитуды и фазы
    orig_amp = np.abs(original)
    orig_phase = np.angle(original)
    trans_amp = np.abs(transformed)
    trans_phase = np.angle(transformed)

    # Замена NaN и Inf на 0
    orig_amp = np.nan_to_num(orig_amp, nan=0.0, posinf=0.0, neginf=0.0)
    trans_amp = np.nan_to_num(trans_amp, nan=0.0, posinf=0.0, neginf=0.0)
    orig_phase = np.nan_to_num(orig_phase, nan=0.0, posinf=0.0, neginf=0.0)
    trans_phase = np.nan_to_num(trans_phase, nan=0.0, posinf=0.0, neginf=0.0)

    # Нормировка для визуализации
    max_val_orig = np.max(orig_amp)
    max_val_trans = np.max(trans_amp)

    # Если максимум 0, избегаем деления на ноль
    if max_val_orig == 0: max_val_orig = 1.0
    if max_val_trans == 0: max_val_trans = 1.0

    # Сохраняем амплитуды
    plt.imsave(f"{output_dir}/{prefix}_original_amp.png", orig_amp / max_val_orig, cmap='gray')
    plt.imsave(f"{output_dir}/{prefix}_transformed_amp.png", trans_amp / max_val_trans, cmap='gray')

    # Сохраняем фазы (всегда от -pi до pi)
    plt.imsave(f"{output_dir}/{prefix}_original_phase.png", orig_phase, cmap='hsv', vmin=-np.pi, vmax=np.pi)
    plt.imsave(f"{output_dir}/{prefix}_transformed_phase.png", trans_phase, cmap='hsv', vmin=-np.pi, vmax=np.pi)

    print(f"  ✓ Сохранено 4 изображения в '{output_dir}': {prefix}_*.png")

def save_gyrator_gif(beam, gyrator, output_dir="results", prefix="result", fps=2):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    frac_pi = np.linspace(0.00, 0.50, 51)
    alphas = frac_pi * np.pi

    print("  Предварительный расчет 51 преобразования...")
    transformed_beams = [gyrator.transform(beam, a) for a in alphas]
    print("  Расчет завершен. Генерация GIF...")

    fig_amp, ax_amp = plt.subplots(figsize=(6, 6))
    all_amp = [np.abs(t) for t in transformed_beams]
    vmax_amp = max(np.max(a) for a in all_amp)
    im_amp = ax_amp.imshow(all_amp[0], cmap='gray', vmin=0, vmax=vmax_amp)
    ax_amp.set_title('Амплитуда')
    ax_amp.axis('off')
    fig_amp.colorbar(im_amp, ax=ax_amp, fraction=0.046, pad=0.04)

    def update_amp(frame_idx):
        im_amp.set_data(all_amp[frame_idx])
        ax_amp.set_title(f'Амплитуда\nα/π = {frac_pi[frame_idx]:.2f}')
        return [im_amp]

    fig_phase, ax_phase = plt.subplots(figsize=(6, 6))
    all_phase = [np.angle(t) for t in transformed_beams]
    im_phase = ax_phase.imshow(all_phase[0], cmap='hsv', vmin=-np.pi, vmax=np.pi)
    ax_phase.set_title('Фаза')
    ax_phase.axis('off')
    fig_phase.colorbar(im_phase, ax=ax_phase, fraction=0.046, pad=0.04)

    def update_phase(frame_idx):
        im_phase.set_data(all_phase[frame_idx])
        ax_phase.set_title(f'Фаза\nα/π = {frac_pi[frame_idx]:.2f}')
        return [im_phase]

    ani_amp = animation.FuncAnimation(fig_amp, update_amp, frames=51, blit=False)
    ani_phase = animation.FuncAnimation(fig_phase, update_phase, frames=51, blit=False)

    amp_path = os.path.join(output_dir, f"{prefix}_amp.gif")
    phase_path = os.path.join(output_dir, f"{prefix}_phase.gif")

    try:
        ani_amp.save(amp_path, writer='pillow', fps=fps)
        ani_phase.save(phase_path, writer='pillow', fps=fps)
        print(f"  ✓ Сохранены GIF: {amp_path}, {phase_path}")
    except Exception as e:
        print(f"  ⚠ Ошибка сохранения GIF. Убедитесь, что установлена библиотека Pillow: {e}")
    finally:
        plt.close('all')

def check_unitarity(original, transformed):
    energy_original = np.sum(np.abs(original)**2)
    energy_transformed = np.sum(np.abs(transformed)**2)
    if energy_original > 0:
        energy_ratio = energy_transformed / energy_original
    else:
        energy_ratio = 1.0
    print(f"\nПроверка унитарности:")
    print(f"  Энергия исходного: {energy_original:.6e}")
    print(f"  Энергия преобразованного: {energy_transformed:.6e}")
    print(f"  Отношение энергий: {energy_ratio:.6f}")
    if abs(energy_ratio - 1.0) < 1e-2:
        print("  ✓ Преобразование унитарно (сохраняет энергию)")
        return True
    else:
        print(f"  ⚠ Преобразование не унитарно (отклонение: {abs(energy_ratio-1.0):.2e})")
        return False

# ============================================================================
# ФУНКЦИИ ВЫБОРА И КОНФИГУРАЦИИ
# ============================================================================
def select_function(function_number):
    print(f"\n{'='*60}")
    print(f"ВЫБОР ФУНКЦИИ #{function_number}")
    print("="*60)
    print("Доступные функции:")
    print("   [1] Гауссов пучок с кубической фазой")
    print("   [2] Гауссов пучок с квадратичной фазой (линза)")
    print("   [3] Простой гауссов пучок")
    print("   [4] Плоская волна с линейной фазой")
    print("   [5] Волна с гиперболической фазой")
    print("   [6] Сферическая волна")
    print("   [7] Мода Эрмита-Гаусса")
    print("   [8] Круговая функция")
    print("   [9] Мода Гаусса–Лагерра")
    print("   [10] Пучок Бесселя (недифрагирующий)")
    print("   [11] Суперпозиция пучков")
    choice = input("   Ваш выбор (1-11): ").strip()
    return choice

def configure_function(choice, size, w0_default=30):
    center = (size // 2, size // 2)

    # --- Стандартные пучки ---
    if choice == '1':
        w0 = float(input(f"   Радиус w0 (по умолч. {w0_default}): ") or w0_default)
        a = float(input("   Коэфф. a (по умолч. 0.0005): ") or "0.0005")
        b = float(input("   Коэфф. b (по умолч. 0.0005): ") or "0.0005")
        func = lambda: gaussian_beam_with_cubic_phase(size, w0, a, b, center)
        params_str = f"Гауссов кубический: w0={w0}, a={a}, b={b}"
    elif choice == '2':
        w0 = float(input(f"   Радиус w0 (по умолч. {w0_default}): ") or w0_default)
        c = float(input("   Коэфф. c (по умолч. 0.001): ") or "0.001")
        d = float(input("   Коэфф. d (по умолч. 0.001): ") or "0.001")
        func = lambda: gaussian_beam_with_quadratic_phase(size, w0, c, d, center)
        params_str = f"Гауссов квадратичный: w0={w0}, c={c}, d={d}"
    elif choice == '3':
        w0 = float(input(f"   Радиус w0 (по умолч. {w0_default}): ") or w0_default)
        func = lambda: simple_gaussian_beam(size, w0, center)
        params_str = f"Простой Гауссов: w0={w0}"
    elif choice == '4':
        kx = float(input("   kx (по умолч. 0.1): ") or "0.1")
        ky = float(input("   ky (по умолч. 0.1): ") or "0.1")
        func = lambda: plane_wave_with_linear_phase(size, kx, ky, center)
        params_str = f"Плоская волна: kx={kx}, ky={ky}"
    elif choice == '5':
        c = float(input("   Коэфф. c (по умолч. 0.01): ") or "0.01")
        func = lambda: hyperbolic_phase_wave(size, c, center)
        params_str = f"Гиперболическая фаза: c={c}"
    elif choice == '6':
        b = float(input("   Коэфф. b (по умолч. 0.001): ") or "0.001")
        func = lambda: spherical_wave(size, b, center)
        params_str = f"Сферическая волна: b={b}"
    elif choice == '7':
        m = int(input("   Индекс m (по умолч. 2): ") or "2")
        n = int(input("   Индекс n (по умолч. 1): ") or "1")
        w0 = float(input(f"   Радиус w0 (по умолч. {w0_default}): ") or w0_default)
        func = lambda: hermit_gaussian_mode(size, m, n, w0, center)
        params_str = f"Эрмит-Гаусс HG_{{{m},{n}}}, w0={w0}"
    elif choice == '8':
        radius = float(input("   Радиус круга (по умолч. 15): ") or "15")
        func = lambda: circle_function(size, radius, center)
        params_str = f"Круг: радиус={radius}"
    elif choice == '9':
        p = int(input("   Индекс p (по умолч. 0): ") or "0")
        l = int(input("   Индекс l (по умолч. 1): ") or "1")
        w0 = float(input(f"   Радиус w0 (по умолч. {w0_default}): ") or w0_default)
        func = lambda: laguerre_gaussian_mode(size, p, l, w0, center)
        params_str = f"Гаусс-Лагерр LG_{{{p},{l}}}, w0={w0}"
    elif choice == '10':
        # Пучок Бесселя
        kr = float(input("   Радиальное волновое число kr (по умолч. 0.2): ") or "0.2")
        func = lambda: bessel_beam(size, kr, center)
        params_str = f"Пучок Бесселя J0: kr={kr}"

    elif choice == '11':
        # Суперпозиция
        n_beams = int(input("   Сколько пучков сложить? (2, 3...): ") or "2")
        beams = []
        weights = []
        combined_params = []

        print("\n   --- Настройка компонентов суперпозиции ---")
        for k in range(n_beams):
            print(f"\n   > Компонент #{k+1}")
            # Рекурсивно используем логику выбора для генерации каждого пучка
            # Для простоты интерфейса можно вызвать configure_function, но
            # проще попросить тип и сконфигурировать его вручную здесь,
            # чтобы не вводить пользователя в заблуждение номерами функций.
            # Однако, переиспользование кода надежнее.
            # Чтобы не печатать полное меню каждый раз, сделаем мини-меню.

            print("   Типы для компонента: [3]Гаусс, [4]Плоская, [7] Мода Эрмита-Гаусса, [9]Лагерр-Гаусс, [10]Бессель")
            sub_type = input("   Выбор: ").strip()

            # Генерируем пучок на основе подвыбора
            if sub_type == '3':
                w0 = float(input(f"   w0 (по умолч. {w0_default}): ") or w0_default)
                beams.append(simple_gaussian_beam(size, w0, center))
                combined_params.append(f"Гаусс(w0={w0})")
            elif sub_type == '4':
                kx = float(input("   kx: ") or "0.1")
                ky = float(input("   ky: ") or "0.1")
                beams.append(plane_wave_with_linear_phase(size, kx, ky, center))
                combined_params.append(f"Плоская(kx={kx})")
            elif sub_type == '7':
                m = int(input("   Индекс m (по умолч. 2): ") or "2")
                n = int(input("   Индекс n (по умолч. 1): ") or "1")
                w0 = float(input(f"   Радиус w0 (по умолч. {w0_default}): ") or w0_default)
                beams.append(hermit_gaussian_mode(size, m, n, w0, center))
                combined_params.append(f"Индекс m={m}, Индекс n={n}")
            elif sub_type == '9':
                p = int(input("   p: ") or "0")
                l = int(input("   l: ") or "1")
                w0 = float(input(f"   w0: ") or w0_default)
                beams.append(laguerre_gaussian_mode(size, p, l, w0, center))
                combined_params.append(f"LG({p},{l})")
            elif sub_type == '10':
                kr = float(input("   kr: ") or "0.2")
                beams.append(bessel_beam(size, kr, center))
                combined_params.append(f"Бессель(kr={kr})")
            else:
                print("   Тип не найден, добавлен простой гауссов пучок.")
                beams.append(simple_gaussian_beam(size, w0_default, center))
                combined_params.append("Гаусс(def)")

            w_str = input("   Вес (комплексное число, напр. 1.0 или 1j): ") or "1"
            try:
                weights.append(complex(w_str))
            except ValueError:
                weights.append(complex(1))
                print("   Ошибка парсинга веса, установлен 1")

        # Создаем суперпозицию
        superposed_result = superpose_beams(beams, weights)

        # Возвращаем функцию-замыкание, возвращающую готовый массив
        func = lambda: superposed_result
        params_str = f"Суперпозиция [{', '.join(combined_params)}]"

    else:
        print("   Неверный выбор, используется Гауссов пучок.")
        w0 = w0_default
        func = lambda: simple_gaussian_beam(size, w0, center)
        params_str = f"Простой Гауссов: w0={w0}"

    return func, params_str

def select_visualization_type():
    print("\nВыбор типа визуализации:")
    print("   [1] Обычная визуализация (амплитуда и фаза)")
    print("   [2] 12 углов (амплитуды и фазы для 12 углов от 0 до π/2)")
    print("   [3] GIF анимация (51 кадр, α/π от 0.00 до 0.50)")
    choice = input("   Ваш выбор (1, 2 или 3): ").strip()
    if choice == '1': return ['standard']
    elif choice == '2': return ['12_angles']
    elif choice == '3': return ['gif']
    else:
        print("   Неверный выбор, используется обычная визуализация")
        return ['standard']

# ============================================================================
# ОСНОВНАЯ ФУНКЦИЯ
# ============================================================================
def main():
    print("=" * 80)
    print("ГИРАТОРНОЕ ПРЕОБРАЗОВАНИЕ ДЛЯ РАЗЛИЧНЫХ ФУНКЦИЙ")
    print("=" * 80)
    size = 512 #512 64
    w0_default = 100
    print("\n СОЗДАНИЕ ГИРАТОРА")
    print(f"   Размер сетки: {size}x{size}")
    gyrator = Gyrator(size=size, scale=2.0)
    print("  Гиратор создан")
    print("\n   Сколько функций вы хотите преобразовать?")
    num_functions = int(input("   Введите число: ") or "1")
    functions, func_params, vis_types, angles, tangens = [], [], [], [], []

    for i in range(num_functions):
        func_choice = select_function(i+1)
        func, params_str = configure_function(func_choice, size, w0_default)
        functions.append(func)
        func_params.append(params_str)
        vis_type = select_visualization_type()
        vis_types.append(vis_type)

        if '12_angles' in vis_type or 'gif' in vis_type:
            angles.append(0.0)
            tangens.append(0.0)
            continue

        print("\n ПАРАМЕТРЫ ПРЕОБРАЗОВАНИЯ")
        print("   [1] Ввести угол α (в радианах или с pi)")
        print("   [2] Ввести параметр H = tan(α)")
        input_type = input("   Ваш выбор (1 или 2): ").strip()

        if input_type == '1':
            print("\n   Введите угол α:")
            print("   Примеры: '0.25' → 0.25π, 'pi/4' → π/4")
            alpha_str = input("   α = ").strip()
            try:
                alpha = parse_angle_input(alpha_str)
                H = np.tan(alpha)
                print(f"   α = {alpha:.6f} рад = {alpha/np.pi:.3f}π")
                print(f"   H = tan(α) = {H:.6f}")
                angles.append(alpha)
                tangens.append(H)
            except Exception as e:
                print(f"   Ошибка: {e}")
                return
        elif input_type == '2':
            print("\n   Введите параметр H:")
            H_str = input("   H = ").strip()
            try:
                H = parse_H_input(H_str)
                if np.isinf(H): alpha = np.pi/2 * np.sign(H)
                elif H == 0: alpha = 0.0
                else: alpha = np.arctan(H)
                print(f"   H = {H}")
                print(f"   α = arctan(H) = {alpha:.6f} рад = {alpha/np.pi:.3f}π")
                angles.append(alpha)
                tangens.append(H)
            except Exception as e:
                print(f"   Ошибка: {e}")
                return
        else:
            print("\n   Используется значение по умолчанию: H = 0")
            angles.append(0.0)
            tangens.append(0.0)

    print(f"   Количество функций: {len(functions)}")
    for i, params in enumerate(func_params):
        print(f"\n   Функция #{i+1}: ")
        print(f"     {params}")
        print(f"     Тип визуализации: {', '.join(vis_types[i])}")
    print("=" * 80)

    try:
        for i, (func, params_str, vis_type, alpha, H) in enumerate(zip(functions, func_params, vis_types, angles, tangens)):
            print(f"\n{'='*80}")
            print(f"ФУНКЦИЯ #{i+1}: {params_str}")
            print("="*80)
            print("СВОДКА ПАРАМЕТРОВ:")
            print("="*80)
            print(f"   Угол α: {alpha:.6f} рад ({alpha/np.pi:.3f}π)")
            print(f"   Параметр H: {H}")

            print("Генерация пучка...")
            beam = func()
            print("Визуализация исходного пучка...")
            #visualize_beam(beam, f"Функция #{i+1}: {params_str}")

            if '12_angles' in vis_type:
                print("Выполнение гираторного преобразования для 12 углов...")
                visualize_12_angles(beam, gyrator, params_str)
                continue

            if 'gif' in vis_type:
                print("Генерация GIF анимации (51 вариант, α/π от 0.00 до 0.50)...")
                safe_prefix = f"result_func{i+1}".replace(".", "_")
                save_gyrator_gif(beam, gyrator, output_dir="results", prefix=safe_prefix)
                continue

            print("Выполнение гираторного преобразования...")
            transformed = gyrator.transform(beam, alpha)

            safe_prefix = f"result_func{i+1}_H{H:.3f}".replace(".", "_")
            save_beam_data_as_png(beam, transformed, safe_prefix, output_dir="results")

            #check_unitarity(beam, transformed)

        print("\n" + "="*80)
        print("ПРЕОБРАЗОВАНИЕ УСПЕШНО ЗАВЕРШЕНО!")
        print("="*80)
    except Exception as e:
        print(f"\nОШИБКА: {e}")

if __name__ == "__main__":
    main()

ГИРАТОРНОЕ ПРЕОБРАЗОВАНИЕ ДЛЯ РАЗЛИЧНЫХ ФУНКЦИЙ

 СОЗДАНИЕ ГИРАТОРА
   Размер сетки: 512x512
  Гиратор создан

   Сколько функций вы хотите преобразовать?
   Введите число: 3

ВЫБОР ФУНКЦИИ #1
Доступные функции:
   [1] Гауссов пучок с кубической фазой
   [2] Гауссов пучок с квадратичной фазой (линза)
   [3] Простой гауссов пучок
   [4] Плоская волна с линейной фазой
   [5] Волна с гиперболической фазой
   [6] Сферическая волна
   [7] Мода Эрмита-Гаусса
   [8] Круговая функция
   [9] Мода Гаусса–Лагерра
   [10] Пучок Бесселя (недифрагирующий)
   [11] Суперпозиция пучков
   Ваш выбор (1-11): 9
   Индекс p (по умолч. 0): 3
   Индекс l (по умолч. 1): 4
   Радиус w0 (по умолч. 100): 96

Выбор типа визуализации:
   [1] Обычная визуализация (амплитуда и фаза)
   [2] 12 углов (амплитуды и фазы для 12 углов от 0 до π/2)
   [3] GIF анимация (51 кадр, α/π от 0.00 до 0.50)
   Ваш выбор (1, 2 или 3): 3

ВЫБОР ФУНКЦИИ #2
Доступные функции:
   [1] Гауссов пучок с кубической фазой
   [2] Гауссов пучок

KeyboardInterrupt: 